# PRESENTATION RUN GUIDE

**Open:** `transfer_learning_frozen_backbone.ipynb` from this project. Select the Python kernel with TensorFlow and scikit-learn installed.

**Run order:** choose **Run All** for a fresh experiment. It retrains both models; do not use it just to view results if training is not intended. For an existing run, inspect its outputs without rerunning the notebook.

- Sections 1 through 8 explain the project, configure the prepared data, and build the two models.
- Section 9 trains only the classification heads while both backbones remain frozen.
- Section 10 evaluates both models; the decision threshold is selected from validation data only.
- Section 11 compares training histories and test metrics; Section 12 saves and reload-checks complete models.
- Section 13 gives the final summary. Run All retrains both models before evaluation and saving.
- Models are saved under `trained_models/`. Metrics, confusion matrices, curves, histories, and comparison files go under `transfer_learning_results/`.

**Presentation summary:** We compare ImageNet-pretrained DenseNet121 and ResNet50 on pediatric chest X-rays. Each pretrained feature extractor (backbone) stays frozen while its new two-class head learns the NORMAL/PNEUMONIA task. Validation guides early stopping and threshold selection; the test split is reserved for final reporting.

No fixed runtime is promised: it depends on hardware. The most recent notebook run records the actual epoch times in the training outputs.

## 1. Project Objective

Transfer learning reuses visual features learned from a large source dataset and adapts them to this smaller chest X-ray task. ImageNet pretrained weights provide general image features at initialization; the model still learns the pneumonia classification task from this project's training split. A backbone is the pretrained convolutional feature extractor. Freezing it prevents its weights from changing while the new classification head is trained. DenseNet121 connects features across dense blocks, while ResNet50 uses residual skip connections. Both architectures and their ImageNet weights are the notebook's existing choices.

The class order is checked from the dataset folders and must be `NORMAL=0`, `PNEUMONIA=1`. The split and preprocessing are not changed.

## 2. Library Imports

TensorFlow/Keras builds the existing transfer models and input pipeline. Scikit-learn supplies the evaluation metrics; Matplotlib writes presentation plots.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

## 3. Configuration

All model and result paths are relative to the project directory. Both models use the existing head-training epoch limit and learning rate.

In [ ]:
PROJECT_DIR = Path.cwd()
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
HEAD_EPOCHS = 5
HEAD_LEARNING_RATE = 1e-3
AUTOTUNE = tf.data.AUTOTUNE

MODEL_DIR = PROJECT_DIR / "trained_models"
RESULTS_DIR = PROJECT_DIR / "transfer_learning_results"
RESULTS_FOLDERS = {
    "metrics": RESULTS_DIR / "Metrics",
    "confusion": RESULTS_DIR / "Confusion matrices",
    "history": RESULTS_DIR / "Training history plots",
    "roc": RESULTS_DIR / "ROC curves",
    "pr": RESULTS_DIR / "Precision-recall curves",
    "comparison": RESULTS_DIR / "Model comparison",
}
for folder in RESULTS_FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)
print("Project:", PROJECT_DIR)
print("Models:", MODEL_DIR)
print("Results:", RESULTS_DIR)

## 4. Dataset Loading

This keeps the existing `chest_xray_cnn_splits/{train,val,test}` folders and integer labels. Only the training loader shuffles. Validation and test remain unshuffled and unaugmented.

In [ ]:
SPLIT_DIR = None  # Example: Path(r"C:/.../chest_xray_cnn_splits")
if SPLIT_DIR is None:
    candidates = [Path.cwd() / "chest_xray_cnn_splits"]
    for root in (Path.home() / ".cache/kagglehub", Path.cwd()):
        if root.is_dir():
            candidates.extend(root.rglob("chest_xray_cnn_splits"))
    SPLIT_DIR = next((p for p in candidates if all((p / s).is_dir() for s in ("train", "val", "test"))), None)
if SPLIT_DIR is None or not all((Path(SPLIT_DIR) / s).is_dir() for s in ("train", "val", "test")):
    raise FileNotFoundError("Set SPLIT_DIR to the cleaned chest_xray_cnn_splits folder created by the split notebook.")
SPLIT_DIR = Path(SPLIT_DIR)

train_ds = tf.keras.utils.image_dataset_from_directory(SPLIT_DIR / "train", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int")
class_names = train_ds.class_names
if class_names != ["NORMAL", "PNEUMONIA"]:
    raise ValueError(f"Expected class order ['NORMAL', 'PNEUMONIA'], found {class_names}")
val_ds = tf.keras.utils.image_dataset_from_directory(SPLIT_DIR / "val", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False, class_names=class_names, label_mode="int")
test_ds = tf.keras.utils.image_dataset_from_directory(SPLIT_DIR / "test", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False, class_names=class_names, label_mode="int")
if len(class_names) != 2:
    raise ValueError(f"Expected two classes, found {class_names}")
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)
print("Data:", SPLIT_DIR, "| class order:", class_names)

## 5. Data Preprocessing

The existing directory loader resizes images to 224 by 224 and supplies RGB batches. Each architecture uses its matching Keras `preprocess_input` function inside the model. Random flip, rotation, and zoom layers are in the model graph and are active during training only; validation and test images are not augmented.

In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.02, seed=SEED + 1),
        tf.keras.layers.RandomZoom(0.05, seed=SEED + 2),
    ],
    name="training_augmentation",
)
print("Training augmentation: horizontal flip, rotation 0.02, zoom 0.05")

## 6. Class Distribution and Class Weights

Inverse-frequency weights are calculated from training labels only and passed to each model's training call. Validation and test labels do not affect them.

In [ ]:
# Count labels from training batches only and inverse-frequency weight them.
counts = np.zeros(len(class_names), dtype=np.int64)
for _, labels in train_ds:
    counts += np.bincount(labels.numpy().reshape(-1), minlength=len(class_names))
if np.any(counts == 0):
    raise ValueError(f"Training split is missing a class: {dict(zip(class_names, counts.tolist()))}")
class_weight = {i: counts.sum() / (len(class_names) * count) for i, count in enumerate(counts)}
print("Training counts:", dict(zip(class_names, counts.tolist())))
print("Class weights:", class_weight)

## 7. DenseNet121 Architecture

DenseNet121 is initialized with ImageNet weights and `include_top=False`, so its original ImageNet classifier is omitted. A global-average-pooling and dropout head feeds the project's two-class softmax. The backbone starts frozen.

In [ ]:
def make_model(architecture):
    if architecture == "DenseNet121":
        backbone = tf.keras.applications.DenseNet121(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
        preprocess = tf.keras.applications.densenet.preprocess_input
    elif architecture == "ResNet50":
        backbone = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3))
        preprocess = tf.keras.applications.resnet50.preprocess_input
    else:
        raise ValueError(f"Unsupported architecture: {architecture}")
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = augmentation(inputs)
    x = tf.keras.layers.Lambda(preprocess, name="preprocess")(x)
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name=architecture.lower())
    return model, backbone

densenet_model, densenet_backbone = make_model("DenseNet121")
print("DenseNet121 input/output:", densenet_model.input_shape, densenet_model.output_shape)
print("DenseNet121 backbone frozen:", not densenet_backbone.trainable)

## 8. ResNet50 Architecture

ResNet50 uses residual skip connections to support deep feature learning. Like DenseNet121, it uses ImageNet initialization, its own matching preprocessing function, and the same new classification-head design for a fair comparison.

In [ ]:
resnet_model, resnet_backbone = make_model("ResNet50")
print("ResNet50 input/output:", resnet_model.input_shape, resnet_model.output_shape)
print("ResNet50 backbone frozen:", not resnet_backbone.trainable)

models = {
    "DenseNet121": (densenet_model, densenet_backbone),
    "ResNet50": (resnet_model, resnet_backbone),
}

## 9. Frozen Backbone Training

Each model trains its classification head for up to `HEAD_EPOCHS`. The pretrained backbone stays frozen and is checked before, during, and after training. Early stopping selects the head state using validation loss and restores its best weights. Accuracy and loss are recorded for training and validation.

In [ ]:
def verify_frozen_backbone(name, backbone):
    assert backbone.trainable is False, f"{name}: backbone became trainable."
    assert len(backbone.trainable_weights) == 0, f"{name}: backbone has trainable weights."

head_histories = {}
for name, (model, backbone) in models.items():
    verify_frozen_backbone(name, backbone)
    print(f"\nTraining {name} - Frozen Backbone")
    check_frozen_during_training = tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: verify_frozen_backbone(name, backbone)
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(HEAD_LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    head_histories[name] = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=2, restore_best_weights=True
            ),
            check_frozen_during_training,
        ],
        verbose=2,
    )
    verify_frozen_backbone(name, backbone)
    head_weight_count = len(model.trainable_weights)
    head_parameter_count = sum(int(np.prod(weight.shape)) for weight in model.trainable_weights)
    print(
        f"{name}: backbone={backbone.name}, trainable={backbone.trainable}, "
        f"trainable backbone weights={len(backbone.trainable_weights)}, "
        f"trainable classification-head weights={head_weight_count} tensors "
        f"({head_parameter_count:,} parameters)"
    )

## 10. Model Evaluation

Early stopping uses validation loss. The classification threshold is also selected on validation data by maximizing validation F1 for PNEUMONIA. This threshold is then fixed before test evaluation. The test split is not used to select models or tune thresholds. Test loss comes from Keras; thresholded metrics use the validation-selected threshold. ROC and precision-recall curves show threshold-independent test performance.

In [ ]:
validation_labels = np.concatenate([labels.numpy().reshape(-1) for _, labels in val_ds])
test_labels = np.concatenate([labels.numpy().reshape(-1) for _, labels in test_ds])
evaluation_results = {}
roc_curves = {}
pr_curves = {}

for name, (model, _) in models.items():
    validation_scores = model.predict(val_ds, verbose=0)[:, 1]
    val_precision, val_recall, val_thresholds = precision_recall_curve(
        validation_labels, validation_scores
    )
    val_f1 = 2 * val_precision[:-1] * val_recall[:-1] / (
        val_precision[:-1] + val_recall[:-1] + 1e-8
    )
    threshold = float(val_thresholds[int(np.argmax(val_f1))])

    test_loss, test_accuracy_argmax = model.evaluate(test_ds, verbose=0)
    test_scores = model.predict(test_ds, verbose=0)[:, 1]
    test_predictions = (test_scores >= threshold).astype(np.int32)
    matrix = confusion_matrix(test_labels, test_predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    specificity = tn / (tn + fp) if tn + fp else 0.0
    test_fpr, test_tpr, _ = roc_curve(test_labels, test_scores)
    test_precision, test_recall, _ = precision_recall_curve(test_labels, test_scores)

    metrics = {
        "model": name,
        "validation_f1_threshold": threshold,
        "test_loss": float(test_loss),
        "test_accuracy": float(accuracy_score(test_labels, test_predictions)),
        "test_accuracy_argmax": float(test_accuracy_argmax),
        "precision": float(precision_score(test_labels, test_predictions, zero_division=0)),
        "recall_sensitivity": float(recall_score(test_labels, test_predictions, zero_division=0)),
        "specificity": float(specificity),
        "f1_score": float(f1_score(test_labels, test_predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(test_labels, test_scores)),
        "average_precision": float(average_precision_score(test_labels, test_scores)),
    }
    evaluation_results[name] = metrics
    roc_curves[name] = (test_fpr, test_tpr, metrics["roc_auc"])
    pr_curves[name] = (test_recall, test_precision, metrics["average_precision"])

    report = classification_report(
        test_labels, test_predictions, labels=[0, 1], target_names=class_names,
        digits=4, zero_division=0,
    )
    report_path = RESULTS_FOLDERS["metrics"] / f"{name}_classification_report.txt"
    report_path.write_text(report, encoding="utf-8")
    print(f"\n{name}: validation-selected threshold={threshold:.4f}")
    print(f"Test loss={test_loss:.4f}; accuracy={metrics['test_accuracy']:.4f}; "
          f"precision={metrics['precision']:.4f}; recall/sensitivity={metrics['recall_sensitivity']:.4f}; "
          f"specificity={specificity:.4f}; F1={metrics['f1_score']:.4f}; "
          f"ROC-AUC={metrics['roc_auc']:.4f}")
    print(report)
    print("Confusion matrix (rows=true, columns=predicted):\n", matrix)

    with (RESULTS_FOLDERS["confusion"] / f"{name}_confusion_matrix.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["true_label", *class_names])
        for true_name, row in zip(class_names, matrix):
            writer.writerow([true_name, *row.tolist()])
    fig, axis = plt.subplots(figsize=(5, 4))
    axis.imshow(matrix, cmap="Blues")
    axis.set(title=f"{name} confusion matrix", xlabel="Predicted label", ylabel="True label",
             xticks=[0, 1], yticks=[0, 1], xticklabels=class_names, yticklabels=class_names)
    for (row, column), value in np.ndenumerate(matrix):
        axis.text(column, row, str(value), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(RESULTS_FOLDERS["confusion"] / f"{name}_confusion_matrix.png", dpi=140)
    display(fig)
    plt.close(fig)

metric_fields = list(next(iter(evaluation_results.values())))
with (RESULTS_FOLDERS["metrics"] / "test_metrics.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=metric_fields)
    writer.writeheader()
    writer.writerows(evaluation_results.values())

fig, axis = plt.subplots(figsize=(7, 6))
for name, (fpr, tpr, auc_value) in roc_curves.items():
    axis.plot(fpr, tpr, label=f"{name} (AUC={auc_value:.3f})")
axis.plot([0, 1], [0, 1], "k--", label="Chance")
axis.set(title="Test ROC curves", xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1))
axis.legend(loc="lower right")
fig.tight_layout()
fig.savefig(RESULTS_FOLDERS["roc"] / "test_roc_curves.png", dpi=150)
display(fig)
plt.close(fig)

fig, axis = plt.subplots(figsize=(7, 6))
for name, (recall_values, precision_values, ap) in pr_curves.items():
    axis.plot(recall_values, precision_values, label=f"{name} (AP={ap:.3f})")
axis.set(title="Test precision-recall curves", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
axis.legend(loc="lower left")
fig.tight_layout()
fig.savefig(RESULTS_FOLDERS["pr"] / "test_precision_recall_curves.png", dpi=150)
display(fig)
plt.close(fig)
print("\nMetrics and evaluation plots saved under:", RESULTS_DIR)

## 11. DenseNet121 vs ResNet50 Comparison

Training plots show accuracy and loss for the head-only phase. The comparison chart uses test metrics reported at each model's validation-selected threshold; no test metric is used to tune a threshold.

In [ ]:
for name in models:
    history = head_histories[name].history
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for metric, axis in (("accuracy", axes[0]), ("loss", axes[1])):
        train_values = history[metric]
        val_values = history[f"val_{metric}"]
        epochs = range(1, len(train_values) + 1)
        axis.plot(epochs, train_values, label="Training")
        axis.plot(epochs, val_values, label="Validation")
        axis.set(title=f"{name}: {metric}", xlabel="Epoch", ylabel=metric.capitalize())
        axis.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_FOLDERS["history"] / f"{name}_training_history.png", dpi=140)
    display(fig)
    plt.close(fig)

comparison_fields = [
    "model", "test_accuracy", "precision", "recall_sensitivity", "specificity",
    "f1_score", "roc_auc", "average_precision",
]
comparison_path = RESULTS_FOLDERS["comparison"] / "model_comparison.csv"
with comparison_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=comparison_fields)
    writer.writeheader()
    writer.writerows({key: row[key] for key in comparison_fields} for row in evaluation_results.values())

plot_metrics = ["test_accuracy", "precision", "recall_sensitivity", "specificity", "f1_score", "roc_auc", "average_precision"]
labels_for_plot = ["Accuracy", "Precision", "Recall", "Specificity", "F1", "ROC-AUC", "Avg precision"]
positions = np.arange(len(plot_metrics))
width = 0.36
fig, axis = plt.subplots(figsize=(11, 5))
for index, (name, row) in enumerate(evaluation_results.items()):
    axis.bar(positions + (index - 0.5) * width, [row[key] for key in plot_metrics], width, label=name)
axis.set(title="DenseNet121 vs ResNet50", ylabel="Score", ylim=(0, 1),
         xticks=positions, xticklabels=labels_for_plot)
axis.legend()
fig.tight_layout()
fig.savefig(RESULTS_FOLDERS["comparison"] / "model_comparison.png", dpi=150)
display(fig)
plt.close(fig)
print("Model comparison saved to:", comparison_path)

## 12. Saving Models

Save the complete frozen-backbone models in native Keras format. Each saved file contains its model architecture and trained weights. The custom preprocessing functions are preserved as Keras Lambda references; `gradcam_analysis.ipynb` loads them with the matching DenseNet or ResNet preprocessing function in `custom_objects`.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_paths = {
    "DenseNet121": MODEL_DIR / "DenseNet121_frozen.keras",
    "ResNet50": MODEL_DIR / "ResNet50_frozen.keras",
}
preprocess_functions = {
    "DenseNet121": tf.keras.applications.densenet.preprocess_input,
    "ResNet50": tf.keras.applications.resnet50.preprocess_input,
}
for name, (model, _) in models.items():
    path = model_paths[name]
    model.save(path)
    reloaded = tf.keras.models.load_model(
        path,
        compile=False,
        safe_mode=True,
        custom_objects={"preprocess_input": preprocess_functions[name]},
    )
    if reloaded.input_shape != model.input_shape or reloaded.output_shape != model.output_shape:
        raise ValueError(f"Saved {name} model did not reload with matching shapes.")
    print(f"Verified complete model: {path.resolve()} ({path.stat().st_size:,} bytes)")

## 13. Final Conclusion

The summary below reports held-out test metrics for the two frozen-backbone models. The chosen PNEUMONIA decision threshold was selected on validation data. Check the `Metrics` and `Model comparison` folders for detailed tables and reports. Grad-CAM and performance metrics are research outputs, not clinical validation; this academic prototype is not a diagnostic tool.

In [ ]:
for name, row in evaluation_results.items():
    print(
        f"{name}: accuracy={row['test_accuracy']:.4f}, "
        f"precision={row['precision']:.4f}, recall={row['recall_sensitivity']:.4f}, "
        f"specificity={row['specificity']:.4f}, F1={row['f1_score']:.4f}, "
        f"ROC-AUC={row['roc_auc']:.4f}, threshold={row['validation_f1_threshold']:.4f}"
    )
print("Models:", *(str(path.resolve()) for path in model_paths.values()), sep="\n")
print("Results:", RESULTS_DIR.resolve())